# Notebook 2: Load Emotion Labels into PostgreSQL

Now that we've determined the inter-rater agreement for each group to determine reliable comment IDs, this notebook loads the reliable emotion labels selected in Notebook 1 into your local PostgreSQL Jira database. The goal is to create a new table (`jira_comment_emotion`) that stores the inter-rater agreement scores for each reliable comment ID, then verify that those IDs join correctly to `jira_issue_comment` and `jira_issue_report`.

### Prerequisites

Before running this notebook:

1. **PostgreSQL must be running locally** with the `jira` database loaded from `emotion_dataset_jira.sql`. See the repo README for setup instructions.
2. **Notebook 1 must have been run**: This notebook re-runs the inter-rater agreement computation from Notebook 1 to reproduce the reliable comment ID lists. You can adjust `AGREEMENT_THRESHOLD` below to change which comments are considered reliable.
3. **The CSV files must be accessible** at the paths set in the paths section below in Step 2.

### Step 1: Import dependencies

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from sqlalchemy import create_engine, text

### Step 2: Set configuration

Update the variables below to match your local setup. `AGREEMENT_THRESHOLD` controls which comment IDs are considered reliable enough to load. Adjust this based on what you observed in the histograms in Notebook 1.

In [ ]:
# PostgreSQL connection details
PG_HOST     = "localhost"
PG_PORT     = 5432
PG_USER     = "postgres"
PG_PASSWORD = "ADD_YOUR_PASSWORD_HERE"
PG_DB       = "jira"

# Paths to the CSV files
GROUP1_PATH = os.path.expanduser('~/PATH_TO/archive3/Groups/group1/')
GROUP2_PATH = os.path.expanduser('~/PATH_TO/archive3/Groups/group2/')
GROUP3_PATH = os.path.expanduser('~/PATH_TO/archive3/Groups/group3/')

# Agreement threshold (adjust based on Notebook 1 histograms)
# A comment ID is considered reliable if at least one emotion meets this threshold
AGREEMENT_THRESHOLD = 0.75

### Step 3: Reproduce reliable comment IDs from Notebook 1

We re-run the inter-rater agreement computation here. The same helper functions from Notebook 1 are used.

In [ ]:
def compute_agreement_table(rater_dfs, emotion_cols, id_col='id', n_raters=None):
    combined = pd.concat(rater_dfs, ignore_index=True)
    for col in emotion_cols:
        combined[col] = combined[col].apply(lambda v: 1 if str(v).strip().lower() == 'x' else 0)
    agreement = combined.groupby(id_col)[emotion_cols].sum().reset_index()
    if n_raters:
        agreement[emotion_cols] = agreement[emotion_cols] / n_raters
    return agreement

# --- Group 1 ---
GROUP1_EMOTIONS = ['love', 'joy', 'surprise', 'anger', 'sadness', 'fear']
GROUP1_N_RATERS = 4

group1_files = sorted(glob.glob(os.path.join(GROUP1_PATH, 'Document*.csv')))
group1_dfs = []
for f in group1_files:
    df = pd.read_csv(f, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    keep_cols = ['id'] + [c for c in GROUP1_EMOTIONS if c in df.columns]
    group1_dfs.append(df[keep_cols])

group1_agreement = compute_agreement_table(group1_dfs, GROUP1_EMOTIONS, n_raters=GROUP1_N_RATERS)

# --- Group 2 ---
GROUP2_EMOTIONS = ['love', 'joy', 'sadness']
GROUP2_N_RATERS = 3

group2_files = sorted(glob.glob(os.path.join(GROUP2_PATH, 'emoiton_*.csv')))
group2_dfs = []
for f in group2_files:
    df = pd.read_csv(f, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    keep = ['id'] + [c for c in GROUP2_EMOTIONS if c in df.columns]
    group2_dfs.append(df[keep])

group2_agreement = compute_agreement_table(group2_dfs, GROUP2_EMOTIONS, n_raters=GROUP2_N_RATERS)

# --- Group 3 ---
GROUP3_EMOTIONS = ['love', 'joy', 'anger', 'sadness']
GROUP3_N_RATERS = 3

group3_files = sorted(glob.glob(os.path.join(GROUP3_PATH, 'SentenceWithEmotion_TRAINING_*.csv')))
group3_dfs = []
for f in group3_files:
    df = pd.read_csv(f, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    keep = ['id'] + [c for c in GROUP3_EMOTIONS if c in df.columns]
    group3_dfs.append(df[keep])

group3_all = pd.concat(group3_dfs, ignore_index=True)
for col in GROUP3_EMOTIONS:
    group3_all[col] = group3_all[col].apply(lambda v: 1 if str(v).strip().lower() == 'x' else 0)

group3_sentence = group3_all.groupby('id')[GROUP3_EMOTIONS].sum().reset_index()
group3_sentence[GROUP3_EMOTIONS] = group3_sentence[GROUP3_EMOTIONS] / GROUP3_N_RATERS
group3_sentence['comment_id'] = group3_sentence['id'].apply(lambda x: str(x).split('_')[0])
group3_comment = group3_sentence.groupby('comment_id')[GROUP3_EMOTIONS].mean().reset_index()
group3_comment.rename(columns={'comment_id': 'id'}, inplace=True)

print('Agreement tables reproduced.')
print(f'Group 1: {len(group1_agreement)} comments')
print(f'Group 2: {len(group2_agreement)} comments')
print(f'Group 3: {len(group3_comment)} comments (comment level)')

### Step 4: Select reliable comment IDs

Select comment IDs from each group where at least one emotion meets the agreement threshold. You can adjust `AGREEMENT_THRESHOLD` in Step 2 and re-run from there.

In [ ]:
g1_reliable = group1_agreement[group1_agreement[GROUP1_EMOTIONS].max(axis=1) >= AGREEMENT_THRESHOLD].copy()
g1_reliable['group_number'] = 1

g2_reliable = group2_agreement[group2_agreement[GROUP2_EMOTIONS].max(axis=1) >= AGREEMENT_THRESHOLD].copy()
g2_reliable['group_number'] = 2

g3_reliable = group3_comment[group3_comment[GROUP3_EMOTIONS].max(axis=1) >= AGREEMENT_THRESHOLD].copy()
g3_reliable['group_number'] = 3

print(f'Reliable comment IDs at threshold {AGREEMENT_THRESHOLD}:')
print(f'  Group 1: {len(g1_reliable)}')
print(f'  Group 2: {len(g2_reliable)}')
print(f'  Group 3: {len(g3_reliable)}')
print(f'  Total:   {len(g1_reliable) + len(g2_reliable) + len(g3_reliable)}')

### Step 5: Connect to PostgreSQL and create the `jira_comment_emotion` table

Create a new table to store the emotion label agreement scores for each reliable comment ID.

In [ ]:
engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")

with engine.connect() as con:
    con.execute(text("DROP TABLE IF EXISTS jira_comment_emotion;"))
    con.execute(text("""
        CREATE TABLE jira_comment_emotion (
            comment_id  INTEGER,
            group_number INTEGER,
            love        REAL,
            joy         REAL,
            sadness     REAL,
            anger       REAL,
            surprise    REAL,
            fear        REAL
        );
    """))
    con.commit()

print("Created jira_comment_emotion table.")

### Step 5b: Import `jira_user` table

The `jira_user` table does not load correctly in the bulk SQL import due to encoding issues in the dump file. This step extracts its INSERT statements and loads them separately.

Update `SQL_DUMP_PATH` below to point to your local copy of `emotion_dataset_jira.sql`.

In [ ]:
import subprocess

SQL_DUMP_PATH = os.path.expanduser('~/PATH_TO/archive3/emotion_dataset_jira.sql')

with engine.connect() as con:
    user_count = pd.read_sql(text("SELECT COUNT(*) AS n FROM jira_user;"), con).iloc[0, 0]

if user_count > 0:
    print(f"jira_user already has {user_count} rows — skipping import.")
else:
    print("jira_user is empty — importing from SQL dump...")
    tmp_path = '/tmp/jira_user_only.sql'

    # Extract only jira_user INSERT statements from the dump
    with open(SQL_DUMP_PATH) as f_in, open(tmp_path, 'w') as f_out:
        for line in f_in:
            if line.startswith('INSERT INTO jira_user'):
                f_out.write(line)

    # Load into PostgreSQL
    subprocess.run(
        ['psql', '-U', PG_USER, '-h', PG_HOST, '-p', str(PG_PORT), '-d', PG_DB, '-f', tmp_path],
        env={**os.environ, 'PGPASSWORD': PG_PASSWORD},
        capture_output=True
    )

    with engine.connect() as con:
        user_count = pd.read_sql(text("SELECT COUNT(*) AS n FROM jira_user;"), con).iloc[0, 0]

    print(f"Import complete. jira_user now has {user_count} rows.")

### Step 6: Insert emotion labels into the table

Combine the reliable comment IDs from all three groups and insert them into `jira_comment_emotion`.

In [ ]:
ALL_EMOTION_COLS = ['love', 'joy', 'sadness', 'anger', 'surprise', 'fear']

def prepare_for_insert(df, group_num):
    out = pd.DataFrame()
    out['comment_id'] = df['id'].astype(int)
    out['group_number'] = group_num
    for col in ALL_EMOTION_COLS:
        out[col] = df[col] if col in df.columns else None
    return out

g1_insert = prepare_for_insert(g1_reliable, 1)
g2_insert = prepare_for_insert(g2_reliable, 2)
g3_insert = prepare_for_insert(g3_reliable, 3)

all_labels = pd.concat([g1_insert, g2_insert, g3_insert], ignore_index=True)

all_labels.to_sql('jira_comment_emotion', engine, if_exists='append', index=False)

print(f'Inserted {len(all_labels)} rows into jira_comment_emotion.')
print(f'  Group 1: {len(g1_insert)} rows')
print(f'  Group 2: {len(g2_insert)} rows')
print(f'  Group 3: {len(g3_insert)} rows')

### Step 7: Verify the load

Check that the expected number of rows loaded and that there are no duplicate comment IDs within the same group.

In [ ]:
with engine.connect() as con:
    total = pd.read_sql(text("SELECT COUNT(*) AS total FROM jira_comment_emotion;"), con)
    per_group = pd.read_sql(text("SELECT group_number, COUNT(*) AS row_count FROM jira_comment_emotion GROUP BY group_number ORDER BY group_number;"), con)
    duplicates = pd.read_sql(text("""
        SELECT group_number, COUNT(*) AS duplicate_count
        FROM (
            SELECT comment_id, group_number, COUNT(*) AS n
            FROM jira_comment_emotion
            GROUP BY comment_id, group_number
            HAVING COUNT(*) > 1
        ) dupes
        GROUP BY group_number;
    """), con)

print(f"Total rows loaded: {total['total'].iloc[0]}")
print()
print("Rows per group:")
display(per_group)

if len(duplicates) == 0:
    print("PASS: No duplicate comment IDs within any group.")
else:
    print("WARNING: Duplicate comment IDs found:")
    display(duplicates)

### Step 8: Test the INNER JOIN to `jira_issue_comment` and `jira_issue_report`

Join the emotion labels to `jira_issue_comment` via `comment_id`, then to `jira_issue_report` to get project context across all communities (Apache, Spring, JBoss, CodeHaus).

In [ ]:
join_query = """
SELECT
    e.comment_id,
    e.group_number,
    e.love,
    e.joy,
    e.sadness,
    e.anger,
    e.surprise,
    e.fear,
    c.comment,
    c.creationdate,
    r.project_name,
    r.repositoryname,
    r.key AS issue_key
FROM jira_comment_emotion e
INNER JOIN jira_issue_comment c ON e.comment_id = c.id
INNER JOIN jira_issue_report r ON c.issue_report_id = r.id
ORDER BY e.group_number, e.comment_id;
"""

with engine.connect() as con:
    joined = pd.read_sql(text(join_query), con)

print(f'Rows returned from INNER JOIN (all communities): {len(joined)}')
print(f'Unique comment IDs matched: {joined["comment_id"].nunique()}')
print(f'Communities represented: {joined["repositoryname"].nunique()}')
print(f'Projects represented: {joined["project_name"].nunique()}')
display(joined.head(10))

### Step 9: Preview the final joined output

This shows a summary of which projects have emotion-labeled comments broken down by community. It also shows a sample of the final joined dataset.

In [ ]:
print("Projects with emotion-labeled comments by community:")
project_summary = (
    joined.groupby(['repositoryname', 'project_name'])['comment_id']
    .count()
    .reset_index()
    .rename(columns={'comment_id': 'labeled_comment_count'})
    .sort_values(['repositoryname', 'labeled_comment_count'], ascending=[True, False])
)
display(project_summary)